# 🌸 SaralGati: Fine-Tune Meta-Llama-3.1-8B-Instruct for Cloudflare Workers AI

This notebook trains a custom **LoRA Adapter** on **Meta-Llama-3.1-8B-Instruct** using the **1,000 multi-turn SaralGati Elder-Guidance dataset**.

### ⚡ Hardware Requirement:
- Google Colab **Free T4 GPU** (Go to `Runtime` -> `Change runtime type` -> select `T4 GPU`).
- Total training time: **~12 to 14 minutes**.

### 🎯 Output:
1. Creates `saralgati_llama31_8b_lora.zip` and auto-downloads to your computer.
2. (Optional) 1-Click direct upload to your **Cloudflare Workers AI** account!

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load Meta-Llama-3.1-8B-Instruct & Configure LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Setup LoRA for Cloudflare compatibility
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ Llama-3.1-8B model & LoRA adapter initialized successfully!")

## 3. Load & Prepare SaralGati 1,000 Multi-Turn Dataset

In [ ]:
import json
import urllib.request
from datasets import Dataset

# Download official SaralGati multi-turn dataset from GitHub repository
dataset_url = "https://raw.githubusercontent.com/ShunyaPulse/SaralGati/main/data/saralgati_multiturn_train.jsonl"
urllib.request.urlretrieve(dataset_url, "saralgati_multiturn_train.jsonl")

data = []
with open("saralgati_multiturn_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"✅ Successfully loaded {len(data)} multi-turn samples from GitHub repository!")

# Format using official Llama-3.1 chat template
def format_prompts(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)
print("Sample Formatted Text:\n", dataset[0]["text"][:300], "...")

## 4. Train Model (~12 minutes on free T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 125, # Full 1-epoch pass on 1000 samples
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting training on GPU...")
trainer_stats = trainer.train()
print("🎉 Training completed successfully!")

## 5. Export Adapter & Patch for Cloudflare Workers AI

In [ ]:
import json
import os
import shutil

output_dir = "saralgati_llama31_8b_lora"
os.makedirs(output_dir, exist_ok = True)

# Save LoRA adapter weights
model.save_pretrained_merged(output_dir, tokenizer, save_method = "lora")

# Patch adapter_config.json with exact base_model_name_or_path for Cloudflare Workers AI
config_path = os.path.join(output_dir, "adapter_config.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

config["base_model_name_or_path"] = "meta-llama/Llama-3.1-8B-Instruct"
config["model_type"] = "llama"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

# Zip for download & upload
zip_filename = "saralgati_llama31_8b_lora"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"✅ Created archive: {zip_filename}.zip")
print(f"File size: {os.path.getsize(zip_filename + '.zip') / (1024*1024):.2f} MB")

## 6. Download LoRA Zip to Your Computer

In [ ]:
from google.colab import files
files.download("saralgati_llama31_8b_lora.zip")

## 7. 🚀 1-Click Direct Upload to Cloudflare Workers AI

In [ ]:
import requests
import getpass

# Interactive prompt for Cloudflare Credentials
ACCOUNT_ID = input("Enter Cloudflare Account ID (default: 7953d66fe9e6158b01faf0752ae8c841): ").strip() or "7953d66fe9e6158b01faf0752ae8c841"
API_TOKEN = getpass.getpass("Paste Cloudflare API Token: ").strip()
FINE_TUNE_NAME = "saralgati-elder-llama31-8b"
DESCRIPTION = "SaralGati Elder-Friendly Hindi Companion on Llama-3.1-8B-Instruct"

headers = {
    "Authorization": f"Bearer {API_TOKEN}"
}

print(f"Step 1: Creating fine-tune '{FINE_TUNE_NAME}' on Cloudflare...")
create_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes"
create_payload = {
    "name": FINE_TUNE_NAME,
    "description": DESCRIPTION,
    "model": "@cf/meta/llama-3.1-8b-instruct-fast"
}

res = requests.post(create_url, headers=headers, json=create_payload)
print(f"Create Status: {res.status_code}")
print(res.text)

finetune_id = None
if res.status_code in [200, 201]:
    finetune_id = res.json().get("result", {}).get("id")
else:
    # Check if it already exists
    list_res = requests.get(create_url, headers=headers)
    if list_res.ok:
        for ft in list_res.json().get("result", []):
            if ft.get("name") == FINE_TUNE_NAME:
                finetune_id = ft.get("id")
                print(f"Found existing fine-tune ID: {finetune_id}")
                break

if finetune_id:
    print(f"\nStep 2: Uploading adapter assets to fine-tune '{FINE_TUNE_NAME}' ({finetune_id})...")
    upload_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes/{finetune_id}/finetune-assets"

    files_to_upload = [
        ("files", ("adapter_config.json", open(f"{output_dir}/adapter_config.json", "rb"), "application/json")),
        ("files", ("adapter_model.safetensors", open(f"{output_dir}/adapter_model.safetensors", "rb"), "application/octet-stream"))
    ]

    upload_res = requests.post(upload_url, headers=headers, files=files_to_upload)
    print(f"Upload Status: {upload_res.status_code}")
    print(upload_res.text)
    if upload_res.ok:
        print("\n🎉🎉 SUCCESS! LoRA adapter is fully deployed and ready for inference on Cloudflare Workers AI!")
else:
    print("❌ Could not obtain fine-tune ID. Check API token permissions.")